# Shirudo 정리: OpenAI·Cisco가 공개한 eBPF 런타임으로 재시작 없이 취약점을 임시 차단하는 원리와 VM 퀵스타트 따라하기

위에서부터 차례로 실행해 보세요.

### 의존성 설치 (Debian 12 기준, 소스 빌드 포함)

In [ ]:
%%bash
set -eo pipefail
apt-get update
apt-get install -y pkg-config libelf-dev zlib1g-dev libcurl4-openssl-dev libluajit-5.1-dev wget gnupg lsb-release perl autoconf automake libtool autoconf-archive libjson-c-dev uuid-dev git curl
curl -s https://apt.llvm.org/llvm-snapshot.gpg.key | gpg --dearmor -o /etc/apt/trusted.gpg.d/llvm.gpg
echo 'deb http://apt.llvm.org/bookworm/ llvm-toolchain-bookworm-23 main' > /etc/apt/sources.list.d/llvm23.list
apt-get update
apt-get install -y clang-23 llvm-23 lld-23
ln -sf /usr/bin/clang-23 /usr/bin/clang
mkdir -p /content/dl && cd /content/dl
curl -sL https://github.com/openssl/openssl/releases/download/openssl-3.5.4/openssl-3.5.4.tar.gz -o o.tgz && tar xzf o.tgz && cd openssl-3.5.4
./Configure --prefix=/opt/ossl35 --libdir=lib && make -j8 && make install_sw
cd /content/dl && curl -sL https://github.com/tpm2-software/tpm2-tss/releases/download/4.1.3/tpm2-tss-4.1.3.tar.gz -o t.tgz && tar xzf t.tgz && cd tpm2-tss-4.1.3
export PKG_CONFIG_PATH=/opt/ossl35/lib/pkgconfig LD_LIBRARY_PATH=/opt/ossl35/lib
./configure --prefix=/opt/tss --disable-doxygen-doc --disable-fapi --disable-policy --disable-tcti-spi-helper --disable-tcti-swtpm --disable-tcti-mssim --disable-tcti-libtpms --disable-unit --disable-integration --without-udevrulesdir --without-udevrulesprefix
make -j8 && make install
git clone --depth 1 https://github.com/shirudo-dev/shirudo /content/shirudo

### 커널 번들 받기 + make + make shirudo-build

In [ ]:
%%bash
set -eo pipefail
export PKG_CONFIG_PATH=/opt/ossl35/lib/pkgconfig:/opt/tss/lib/pkgconfig LD_LIBRARY_PATH=/opt/ossl35/lib:/opt/tss/lib PATH=/opt/ossl35/bin:$PATH
cd /content/shirudo
testing/vm/kernel-artifact.sh fetch
make CC=clang -j8 2>&1 | tail -5
make CC=clang shirudo-build 2>&1 | tail -5
ls -l shirudo tools/shirudo-build/shirudo-build
./shirudo version

### 키 생성 → .upatch 빌드·서명 → info (G1 에서 make shirudo-build 로 만든 shirudo 저장소 안에서, OpenSSL 3.5+ 필요)

In [ ]:
%%bash
set -eo pipefail
cd shirudo
export PATH="$PWD:$PWD/tools/shirudo-build:$PATH"
shirudo genkey --keys-dir /tmp/keys
shirudo build testing/fixtures/basic.bpf.c --keys-dir /tmp/keys --vmlinux vendor/linux/vmlinux.h -o basic.upatch
ls -l basic.upatch
shirudo upatch info basic.upatch